# Main experiment: unbalanced WFR for replay-free, task-free continual pretraining

**Question.** When a pretrained language model keeps learning from a stream whose domain changes without announcement, does the unbalanced Wasserstein–Fisher–Rao flow (**U**) forget less than (a) the same model with normalised masses (BAL), (b) parameter-matched LoRA, and (c) a learned-router mixture of LoRAs, without costing plasticity?

**The learners never receive** a domain label, a boundary, or the number of domains, and they store no data (replay, ER-2%, is a labelled reference only). Only *Oracle-experts* sees domain labels; it is an upper reference, not a competitor.

**Model:** Pythia-160M-deduped, frozen, with adapters (Full-FT trains all weights). **Stream:** 7 arXiv categories, AG News world news and German Wikipedia in a seed-specific random order, then the first domain again (recurrence test): 10 segments × 250 steps × 16 × 512 tokens (preliminary G5: 30.4 GPU-hours at 300 steps, 25.8 at 250). Documents are joined with a blank line, not `<|endoftext|>`, which Pythia-deduped scores at about 1,220 nats as a next token (found in the first preliminary run). **Tuning** uses only the development stream (Tier 0); its outcome is frozen in `frozen_settings.json`, and the test tiers refuse to run without it.

| Tier | Conditions | Seeds | Purpose |
|---|---|---|---|
| 0 | LoRA lr, Full-FT lr, U's δ, OEWC λ (3 values each) | 1, dev stream | calibration, then freeze |
| 1 | U, BAL, LoRA-match, LoRA-match-sched, MoLoRA-router, Full-FT, OEWC | 3 | primary hypotheses H1–H3, H5 |
| 2 | U-K4 (bounded size, merging), U-trunkfree, U-nocreate | 3 | H4 and ablations |
| 3 | Oracle-experts, ER-2%, Joint, LoRA-match ×0.5 and ×0.25, Single-domain | 2 (1) | references and the LoRA frontier |
| 4 | U, BAL, LoRA-match on a gradual stream | 1 | robustness to blurred boundaries |

**Runtime.** About 26 GPU-hours (preliminary G5). U's step-based settings (horizons, promotion evidence, BAL's window) are the preliminary experiment's, scaled from 200- to 250-step segments; the creation threshold is preliminary v3's (more than 65% of a batch uncovered). U's E-step runs the 8 bottom blocks once and only the top 4 blocks per component (the same numbers, checked at start-up against full forwards). Colab sessions end after 12–24 hours, so the work spans 2–3 sessions: every finished run is saved to `My Drive/uwfr_main/`, reruns skip it, and `budget()` shows what is left. Within a tier, seeds run outer and conditions inner, so if time runs out every condition has the same seeds.

**Run order:** sections 1–8 (definitions, about 20–30 minutes for the one-time data download), then 9 (Tier 0, about 3 hours), 10 (pre-registration: file the printed hash before going on), then Tiers 1–4, then section 16. After a disconnect: run sections 1–8 again, then continue where you stopped.

In [ ]:
!pip install -q "datasets>=2.18" "transformers>=4.40" scikit-learn scipy

## 1. Setup and configuration
All settings are in `CFG`. The four tuned values (`lr_lora`, `lr_full`, `delta`, `oewc_lambda`) are placeholders here: Tier 0 chooses them and `frozen_settings.json` overrides them. Changing any other setting after Tier 1 has started is refused (`results.json` stores the settings).

In [ ]:
# Main experiment: unbalanced WFR (U) for replay-free, task-free continual pretraining of Pythia-160M.
import os, math, time, json, re, html, random, urllib.request, csv, hashlib, glob, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OFFLINE = os.environ.get("UWFR_OFFLINE") == "1"          # code check only: tiny random model, byte tokenizer, local text
USE_DRIVE = not OFFLINE and not os.environ.get("UWFR_OUT")
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/uwfr_main"
else:
    ROOT = os.environ.get("UWFR_OUT", "uwfr_main")
os.makedirs(ROOT, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| outputs:", ROOT)

CFG = dict(
    model="EleutherAI/pythia-160m-deduped",
    seq_len=512, batch=16, steps_per_segment=250, warmup=50, clip=1.0, amp=True,   # 250: preliminary G5 (300 -> 30.4 GPU-h)
    lr_lora=5e-4, lr_full=3e-5,                       # placeholders: Tier 0 replaces them (frozen_settings.json)
    shared_rank=32, comp_rank=16, comp_layers=4, K_max=12, K_budget=4,
    # unbalanced WFR. Step-based settings are those of the preliminary experiment scaled to the segment length (scaled())
    prefix=128, delta=0.10, level_q=0.75, level_rate=0.05,
    nu_region=0.65,   # > 65% of a batch uncovered: <= 3e-4 false alarms per step at level_q = 0.75 (preliminary v3)
    # baselines
    router_topk=2, aux_coef=0.01,                      # MoLoRA-router (standard values, not tuned)
    oewc_lambda=1e-3, oewc_gamma=0.9, oewc_every=100,  # online EWC (lambda tuned in Tier 0)
    er_fraction=0.02, er_replay=4,                     # ER: reservoir of 2% of stream sequences, 4 of 16 per batch replayed
    gradual_window=0.2,                                # Tier 4: blended share of each segment at its boundary
    separator="\n\n",                                  # between documents (NOT <|endoftext|>: see the data section)
    # data and evaluation
    train_bytes=4_000_000, test_bytes=400_000, eval_windows=24, lambada_n=500, wikitext_windows=24,
    # Tier 0 calibration (development stream)
    cal_steps=150, cal_lr_lora=[2e-4, 5e-4, 1e-3], cal_lr_full=[1e-5, 3e-5, 1e-4],
    cal_delta=[0.05, 0.10, 0.15], cal_oewc=[1e-4, 1e-3, 1e-2],
)
PRELIM_RATIO = dict(A_p=130 / 200, A_s=130 / 200, E_min=70 / 200, E_floor=25 / 200, bal_window=67 / 200)   # preliminary: 200-step segments
def scaled(S): return {k: max(1, round(r * S)) for k, r in PRELIM_RATIO.items()}
CFG.update(scaled(CFG["steps_per_segment"]))
if OFFLINE:
    CFG.update(model="tiny", seq_len=128, batch=8, steps_per_segment=24, prefix=32, shared_rank=8, comp_rank=4, comp_layers=1,
               K_max=6, K_budget=3, train_bytes=200_000, test_bytes=30_000,
               eval_windows=6, cal_steps=10, warmup=5, oewc_every=10, cal_lr_lora=[5e-4, 1e-3], cal_lr_full=[1e-4],
               cal_delta=[0.10], cal_oewc=[1e-3])
    CFG.update(scaled(CFG["steps_per_segment"]))
AMP = CFG["amp"] and DEVICE == "cuda"
def autocast(): return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=torch.float16, enabled=AMP)

TUNED = ["lr_lora", "lr_full", "delta", "oewc_lambda"]
FROZEN_PATH = os.path.join(ROOT, "frozen_settings.json")
def load_frozen(required=True):
    if os.path.exists(FROZEN_PATH):
        fz = json.load(open(FROZEN_PATH)); CFG.update({k: fz[k] for k in TUNED if k in fz}); return fz
    if required: raise RuntimeError("No frozen_settings.json: run Tier 0 (section 9) first, or set ALLOW_UNFROZEN = True for a code check.")
    return None
ALLOW_UNFROZEN = OFFLINE

# kind: lora | full | moe | pop.  Options are explained in the methods section.
CONDITIONS = {
    # Tier 1
    "U":                dict(kind="pop", tier=1),
    "BAL":              dict(kind="pop", balanced=True, tier=1),
    "LoRA-match":       dict(kind="lora", tier=1),
    "LoRA-match-sched": dict(kind="lora", schedule=True, tier=1),
    "MoLoRA-router":    dict(kind="moe", tier=1),
    "Full-FT":          dict(kind="full", tier=1),
    "OEWC":             dict(kind="lora", oewc=True, tier=1),
    # Tier 2
    "U-K4":             dict(kind="pop", budget=True, tier=2),
    "U-trunkfree":      dict(kind="pop", trunk_free=True, tier=2),
    "U-nocreate":       dict(kind="pop", nocreate=True, tier=2),
    # Tier 3 (references)
    "Oracle-experts":   dict(kind="pop", oracle=True, tier=3),
    "ER-2%":            dict(kind="lora", replay=True, tier=3),
    "Joint":            dict(kind="lora", joint=True, tier=3),
    "LoRA-match-x0.5":  dict(kind="lora", lr_scale=0.5, tier=3),
    "LoRA-match-x0.25": dict(kind="lora", lr_scale=0.25, tier=3),
    # Tier 4 (gradual stream)
    "U-gradual":        dict(kind="pop", gradual=True, tier=4),
    "BAL-gradual":      dict(kind="pop", balanced=True, gradual=True, tier=4),
    "LoRA-match-gradual": dict(kind="lora", gradual=True, tier=4),
}
SEEDS = {1: [0, 1, 2], 2: [0, 1, 2], 3: [0, 1], 4: [0]}
print(f"{len(CONDITIONS)} conditions; seeds per tier {SEEDS}")

## 2. Data
arXiv abstracts by primary category (streamed from `gfissore/arxiv-abstracts-2021`), AG News world news and German Wikipedia (`wikimedia/wikipedia`, 20231101.de, articles cut to 3,000 characters). About 4 MB of training text and 0.4 MB of held-out text per domain, split by document. The cell prints the number of passes per segment for each domain. Documents are joined with a blank line. The development domains are the preliminary experiment's.

In [ ]:
TEST = {   # main-study domains (never used for tuning)
    "Algebraic geometry": ("arxiv", "math.AG"), "High-energy theory": ("arxiv", "hep-th"),
    "Quantum physics": ("arxiv", "quant-ph"), "Machine learning": ("arxiv", "cs.LG"),
    "Cryptography": ("arxiv", "cs.CR"), "Galaxies": ("arxiv", "astro-ph.GA"),
    "Neuroscience": ("arxiv", "q-bio.NC"), "World news": ("agnews", "World"),
    "German Wikipedia": ("dewiki", None),
}   # 9 domains (+ recurrence = 10 segments); Probability, Statistical methods and Econometrics were cut for the budget
DEV = {    # development stream (Tier 0 only), the same domains as the preliminary experiment
    "Number theory": ("arxiv", "math.NT"), "Strongly corr. el.": ("arxiv", "cond-mat.str-el"),
    "Computer vision": ("arxiv", "cs.CV"), "Quant. finance": ("arxiv", "q-fin."), "Sports news": ("agnews", "Sports"),
}
if OFFLINE:
    TEST = {"World news": ("agnews", "World"), "Business news": ("agnews", "Business"), "Sci/Tech news": ("agnews", "Sci/Tech"),
            "Python code": ("local", "py"), "C headers": ("local", "h")}
    DEV = {"Sports news": ("agnews", "Sports"), "Python code (dev)": ("local", "py"), "Business (dev)": ("agnews", "Business")}
AG_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/{}.csv"
AG_NAMES = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

def _clean(s):
    s = re.sub(r"#(\d+);", lambda m: chr(int(m.group(1))), str(s))
    return re.sub(r"\s+", " ", html.unescape(s).replace("\\", " ")).strip()
_ENT = {"quot": '"', "amp": "&", "lt": "<", "gt": ">", "apos": "'"}
def _fix_ag(s):
    """AG News's CSV has HTML entities without the '&' (' quot;') and stray tags ('<b>...</b>')"""
    s = re.sub(r"\b(quot|amp|lt|gt|apos);", lambda m: _ENT[m.group(1)], s); s = re.sub(r"</?[a-zA-Z]{1,6}>", "", s)
    return re.sub(r"\s+", " ", s).strip()

def primary_category(cats):
    if isinstance(cats, (list, tuple)): cats = " ".join(map(str, cats))
    t = str(cats).split(); return t[0] if t else ""
def need(): return int((CFG["train_bytes"] + CFG["test_bytes"]) * 1.1)

def arxiv_from_hf(specs):
    from datasets import load_dataset
    ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train", streaming=True).shuffle(seed=0, buffer_size=20_000)
    docs = {t: [] for t in specs}; size = {t: 0 for t in specs}; t0 = time.time()
    for n, r in enumerate(ds):
        pc = primary_category(r.get("categories", ""))
        for t, pref in specs.items():
            if size[t] < need() and pc.startswith(pref):
                d = _clean(r.get("title", "")) + ". " + _clean(r.get("abstract", "")); docs[t].append(d); size[t] += len(d.encode()) + 1
                break
        if all(size[t] >= need() for t in specs): break
        if n % 200_000 == 0 and n: print(f"  scanned {n:,} rows ({time.time() - t0:.0f}s): " + ", ".join(f"{t[:10]} {size[t] / 1e6:.1f}" for t in specs))
    return docs

def agnews_docs(classes):
    docs = {c: [] for c in classes}
    for split in ("train", "test"):
        path = os.path.join(ROOT, f"ag_news_{split}.csv")
        if not os.path.exists(path): urllib.request.urlretrieve(AG_URL.format(split), path)
        with open(path, newline="", encoding="utf-8") as f:
            for label, title, desc in csv.reader(f):
                if AG_NAMES[int(label)] in docs: docs[AG_NAMES[int(label)]].append(_fix_ag(_clean(title) + ". " + _clean(desc)))
    return docs

def dewiki_docs():
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.de", split="train", streaming=True).shuffle(seed=0, buffer_size=10_000)
    out, size = [], 0
    for r in ds:
        d = r["text"][:3000]; out.append(d); size += len(d.encode())
        if size >= need(): break
    return out

def local_docs(ext):
    pats = {"py": os.path.dirname(os.__file__) + "/**/*.py", "h": "/usr/include/**/*.h"}
    fs = sorted(glob.glob(pats[ext], recursive=True)); random.Random(0).shuffle(fs); out, size = [], 0
    for f in fs:
        txt = open(f, encoding="utf-8", errors="ignore").read()
        for i in range(0, len(txt), 2000): out.append(txt[i:i + 2000]); size += 2000
        if size > need(): break
    return out

def load_docs():
    cache = os.path.join(ROOT, "docs.json"); want = {**TEST, **DEV}
    docs = json.load(open(cache)) if os.path.exists(cache) else {}
    missing = {t: v for t, v in want.items() if t not in docs}
    if not missing: print("loaded cached documents"); return docs
    ax = {t: v[1] for t, v in missing.items() if v[0] == "arxiv"}
    if ax:
        print(f"downloading arXiv abstracts for {len(ax)} categories (streaming the whole dataset can take 15-20 minutes) ...")
        got = arxiv_from_hf(ax)
        for t in ax:
            if len(got[t]) < 300: raise RuntimeError(f"too few documents for {t} ({len(got[t])})")
        docs.update(got)
    ag = [v[1] for v in missing.values() if v[0] == "agnews"]
    if ag:
        a = agnews_docs(ag)
        for t, v in missing.items():
            if v[0] == "agnews": docs[t] = a[v[1]]
    for t, v in missing.items():
        if v[0] == "dewiki": print("downloading German Wikipedia ..."); docs[t] = dewiki_docs()
        if v[0] == "local": docs[t] = local_docs(v[1])
    if not OFFLINE: json.dump(docs, open(cache, "w"))
    return docs

class ByteTok:                                     # offline stand-in only
    eos_token_id = 256
    def encode(self, s): return list(s.encode("utf-8", "ignore"))
    def __call__(self, texts): return {"input_ids": [self.encode(t) for t in texts]}
if OFFLINE: TOK = ByteTok()
else:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(CFG["model"])

# Documents are joined with a blank line, not with <|endoftext|>: Pythia-deduped gives <|endoftext|> as a next token
# a loss of about 1,220 nats (found in the first preliminary run, identical in every domain), which swamped everything.
SEP = TOK(["\n\n"])["input_ids"][0]
print("document separator ids:", SEP)
def encode_docs(docs):
    ids = []
    for i in range(0, len(docs), 512):
        for e in TOK(docs[i:i + 512])["input_ids"]: ids.extend(e); ids.extend(SEP)
    return torch.tensor(ids, dtype=torch.long)

def split_docs(d):
    d = list(d); random.Random(0).shuffle(d); test, size, i = [], 0, 0
    while i < len(d) and size < CFG["test_bytes"]: test.append(d[i]); size += len(d[i].encode()) + 1; i += 1
    train, size = [], 0
    while i < len(d) and size < CFG["train_bytes"]: train.append(d[i]); size += len(d[i].encode()) + 1; i += 1
    return encode_docs(train), encode_docs(test)

DOCS = load_docs(); TRAIN, HELD = {}, {}
for t in list(TEST) + list(DEV):
    TRAIN[t], HELD[t] = split_docs(DOCS[t])
    passes = CFG["steps_per_segment"] * CFG["batch"] * CFG["seq_len"] / max(1, len(TRAIN[t]))
    print(f"{t:20s} {'test' if t in TEST else 'dev':4s} {len(DOCS[t]):>7,} docs | train {len(TRAIN[t]):>9,} tokens ({passes:.1f} passes per segment) | held-out {len(HELD[t]):>8,}")

def eval_windows(data, n, T):
    pos = np.linspace(0, len(data) - T - 2, n).astype(int)
    return torch.stack([data[i:i + T] for i in pos]).to(DEVICE), torch.stack([data[i + 1:i + T + 1] for i in pos]).to(DEVICE)
EVAL = {t: eval_windows(HELD[t], CFG["eval_windows"], CFG["seq_len"]) for t in HELD}

def make_stream(seed, which="test"):
    """segments (domain names). test: a seed-specific order of all test domains, then the first again (recurrence)."""
    doms = list(TEST if which == "test" else DEV)
    if which == "test":
        order = random.Random(1000 + seed).sample(doms, len(doms)); return order + [order[0]]
    return doms

def batch_for(segs, step, S, gen, mode="hard"):
    """sequences for one step; also each sequence's true domain (used only by the oracle reference and for analysis)"""
    B, T = CFG["batch"], CFG["seq_len"]; s = min(step // S, len(segs) - 1)
    if mode == "joint":
        uniq = list(dict.fromkeys(segs)); doms = [uniq[int(i)] for i in torch.randint(len(uniq), (B,), generator=gen)]
    elif mode == "gradual" and s + 1 < len(segs) and step % S >= S * (1 - CFG["gradual_window"] / 2):
        f = (step % S - S * (1 - CFG["gradual_window"] / 2)) / (S * CFG["gradual_window"])          # 0 -> 0.5 before the boundary
        doms = [segs[s + 1] if float(torch.rand(1, generator=gen)) < f else segs[s] for _ in range(B)]
    elif mode == "gradual" and s > 0 and step % S < S * CFG["gradual_window"] / 2:
        f = 0.5 + (step % S) / (S * CFG["gradual_window"])                                            # 0.5 -> 1 after it
        doms = [segs[s] if float(torch.rand(1, generator=gen)) < f else segs[s - 1] for _ in range(B)]
    else: doms = [segs[s]] * B
    xs, ys = [], []
    for d in doms:
        i = int(torch.randint(len(TRAIN[d]) - T - 1, (1,), generator=gen)); xs.append(TRAIN[d][i:i + T]); ys.append(TRAIN[d][i + 1:i + T + 1])
    return torch.stack(xs).to(DEVICE), torch.stack(ys).to(DEVICE), doms

## 3. Model with shared and component adapters
Frozen Pythia-160M. A shared LoRA (rank 32) on every linear layer; components (rank 16) on the attention output and MLP of the top 4 blocks, at most 12. LoRA-match's rank is set so its trainable parameters equal U's at 12 components. `set_from_deltas` re-factorises a merged update by truncated SVD (used only by U-K4).

In [ ]:
class _Ctx:
    gate = None; aux = []
CTX = _Ctx()

class _Stop(Exception): pass
def _rep(v, B, n):
    """repeat batch-shaped tensors (dim >= 2, leading size B) n times; leave everything else"""
    if torch.is_tensor(v) and v.dim() >= 2 and v.shape[0] == B: return v.repeat(n, *([1] * (v.dim() - 1)))
    if isinstance(v, (tuple, list)): return type(v)(_rep(u, B, n) for u in v)
    return v

class AdaptedLinear(nn.Module):
    """frozen base linear + shared LoRA + K component LoRAs; component k adds ((x A_k) * g_k) B_k"""
    def __init__(s, base, shared_r, K, comp_r, router_topk=0):
        super().__init__(); s.base = base; fin, fout = base.in_features, base.out_features; s.K, s.topk = K, router_topk
        s.sA = nn.Parameter(torch.randn(fin, shared_r) / math.sqrt(fin)) if shared_r else None
        s.sB = nn.Parameter(torch.zeros(shared_r, fout)) if shared_r else None
        if K: s.cA = nn.Parameter(torch.randn(K, fin, comp_r) / math.sqrt(fin)); s.cB = nn.Parameter(torch.zeros(K, comp_r, fout))
        s.router = nn.Linear(fin, K) if (K and router_topk) else None
    def forward(s, x):
        y = s.base(x)
        if s.sA is not None: y = y + (x @ s.sA.to(x.dtype)) @ s.sB.to(x.dtype)
        if s.K:
            if s.router is not None:
                p = s.router(x).float().softmax(-1); top = p.topk(s.topk, -1)
                g = torch.zeros_like(p).scatter(-1, top.indices, top.values / top.values.sum(-1, keepdim=True))
                CTX.aux.append(s.K * (F.one_hot(top.indices[..., 0], s.K).float().mean((0, 1)) * p.mean((0, 1))).sum())
            else: g = CTX.gate[:, None, :]
            h = torch.einsum("bti,kir->btkr", x, s.cA.to(x.dtype)) * g.to(x.dtype)[..., None]
            y = y + torch.einsum("btkr,kro->bto", h, s.cB.to(x.dtype))
        return y

def load_base(name):
    if OFFLINE:
        from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
        m = GPTNeoXForCausalLM(GPTNeoXConfig(vocab_size=257, hidden_size=96, num_hidden_layers=3, num_attention_heads=4,
                                             intermediate_size=384, max_position_embeddings=512))
        torch.manual_seed(1234)
        for p in m.parameters():
            if p.dim() > 1: nn.init.normal_(p, 0, 0.02)
    else:
        from transformers import AutoModelForCausalLM
        try: m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
        except TypeError: m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)
    m.config.use_cache = False
    for p in m.parameters(): p.requires_grad_(False)
    return m.to(DEVICE)

class AdaptedLM(nn.Module):
    """Pythia (GPT-NeoX): shared LoRA on every linear layer of every block; K components on the attention output and MLP
    of the top comp_layers blocks."""
    def __init__(s, shared_r, K=0, comp_r=0, comp_layers=0, router_topk=0):
        super().__init__(); s.lm = load_base(CFG["model"]); s.K = K; layers = s.lm.gpt_neox.layers; L = len(layers)
        s.c0 = L - comp_layers if (K and comp_layers) else L; s.split_ok = None
        for i, blk in enumerate(layers):
            comp = K if i >= L - comp_layers else 0
            for parent, attr in ((blk.attention, "query_key_value"), (blk.attention, "dense"), (blk.mlp, "dense_h_to_4h"), (blk.mlp, "dense_4h_to_h")):
                k = 0 if attr == "query_key_value" else comp
                setattr(parent, attr, AdaptedLinear(getattr(parent, attr), shared_r, k, comp_r, router_topk if k else 0).to(DEVICE))
    def adapters(s): return [m for m in s.modules() if isinstance(m, AdaptedLinear)]
    def expert_params(s): return [p for a in s.adapters() if a.K for p in (a.cA, a.cB)]
    def trunk_params(s):
        out = []
        for a in s.adapters():
            if a.sA is not None: out += [a.sA, a.sB]
            if a.router is not None: out += list(a.router.parameters())
        return out
    def n_trainable(s): return sum(p.numel() for p in s.expert_params() + s.trunk_params())
    def seq_nll(s, x, y):
        logits = s.lm(input_ids=x).logits
        return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), y.reshape(-1), reduction="none").view(y.shape).mean(1)
    @torch.no_grad()
    def clone(s, src, dst):
        for p in s.expert_params(): p[dst] = p[src]
    @torch.no_grad()
    def nll_parts_full(s, x, y, parts, chunk):
        """[len(parts), B] mean NLL of each sequence under each component: one full forward per component"""
        B = x.shape[0]; out = []
        for i in range(0, len(parts), chunk):
            pc = parts[i:i + chunk]; n = len(pc)
            CTX.gate = F.one_hot(torch.tensor(pc, device=DEVICE).repeat_interleave(B), s.K).float()
            out.append(s.seq_nll(x.repeat(n, 1), y.repeat(n, 1)).view(n, B))
        return torch.cat(out)
    @torch.no_grad()
    def nll_parts(s, x, y, parts, chunk):
        """Same numbers as nll_parts_full, cheaper: the blocks below the first component block do not depend on the
        gate, so they run once and only the top blocks run per component. The inputs of the first component block are
        captured with a pre-hook (works across transformers versions); verified against nll_parts_full at start-up."""
        if s.split_ok is False: return s.nll_parts_full(x, y, parts, chunk)
        B = x.shape[0]; layers = s.lm.gpt_neox.layers; store = {}
        def grab(mod, args, kwargs): store["a"], store["k"] = args, dict(kwargs); raise _Stop
        hk = layers[s.c0].register_forward_pre_hook(grab, with_kwargs=True)
        try: s.lm(input_ids=x)
        except _Stop: pass
        finally: hk.remove()
        a, kw = store["a"], store["k"]; hid = a[0] if a else kw.pop("hidden_states"); a = a[1:]; out = []
        for i in range(0, len(parts), chunk):
            pc = parts[i:i + chunk]; n = len(pc)
            CTX.gate = F.one_hot(torch.tensor(pc, device=DEVICE).repeat_interleave(B), s.K).float()
            h = hid.repeat(n, 1, 1); ak = tuple(_rep(v, B, n) for v in a); kk = {k: _rep(v, B, n) for k, v in kw.items()}
            for layer in layers[s.c0:]:
                o = layer(h, *ak, **kk); h = o[0] if isinstance(o, (tuple, list)) else o
            logits = s.lm.get_output_embeddings()(s.lm.gpt_neox.final_layer_norm(h)).float()
            out.append(F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.repeat(n, 1).reshape(-1), reduction="none").view(n, B, -1).mean(2))
        return torch.cat(out)
    @torch.no_grad()
    def check_split(s, x, y):
        """compare the two paths in float32 with two different components; fall back to full forwards on a mismatch"""
        global SPLIT_MSG
        if s.K < 2: s.split_ok = False; return
        saved = [a.cB.data[1].clone() for a in s.adapters() if a.K]
        for a in s.adapters():
            if a.K: a.cB.data[1].normal_(0, 0.02)
        s.split_ok = True; fast = s.nll_parts(x, y, [0, 1], 2); slow = s.nll_parts_full(x, y, [0, 1], 2)
        for a, v in zip([a for a in s.adapters() if a.K], saved): a.cB.data[1] = v
        err = float((fast - slow).abs().max()); s.split_ok = err < 1e-3
        if SPLIT_MSG is None or not s.split_ok:
            SPLIT_MSG = f"shared-bottom E-step: max difference to full forwards {err:.1e} -> " + ("used" if s.split_ok else "NOT used (full forwards)")
            print(SPLIT_MSG)
    @torch.no_grad()
    def deltas(s, k):
        return [(a.cA[k] @ a.cB[k]) for a in s.adapters() if a.K]
    @torch.no_grad()
    def set_from_deltas(s, k, dWs, r):
        """write low-rank factors of the given updates into component k (truncated SVD, rank r)"""
        for a, dW in zip([a for a in s.adapters() if a.K], dWs):
            U_, S_, Vh = torch.linalg.svd(dW.float(), full_matrices=False); sq = S_[:r].sqrt()
            a.cA[k] = U_[:, :r] * sq; a.cB[k] = sq[:, None] * Vh[:r]

SPLIT_MSG = None
def _matched_rank():
    m = AdaptedLM(CFG["shared_rank"], CFG["K_max"], CFG["comp_rank"], CFG["comp_layers"]); total = m.n_trainable()
    per_rank = sum(a.base.in_features + a.base.out_features for a in m.adapters()); del m
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return max(1, round(total / per_rank)), total
MATCH_RANK, U_PARAMS = _matched_rank()
print(f"U at {CFG['K_max']} components: {U_PARAMS / 1e6:.2f}M trainable parameters -> LoRA-match rank {MATCH_RANK}")

## 4. Optimiser
`ParticleAdamW`: one step factor per component and one for the shared adapter. A factor of 0 leaves a component exactly unchanged, including its momentum.

In [ ]:
class ParticleAdamW:
    """Adam with one step factor per component and one for shared parameters; factor 0 = exactly unchanged."""
    def __init__(s, model, betas=(0.9, 0.999), eps=1e-8):
        s.ep, s.tp, s.b1, s.b2, s.eps = model.expert_params(), model.trunk_params(), betas[0], betas[1], eps
        s.m = {id(p): torch.zeros_like(p) for p in s.ep + s.tp}; s.v = {id(p): torch.zeros_like(p) for p in s.ep + s.tp}
        s.t_rows = torch.zeros(max(1, model.K), device=DEVICE); s.t_trunk = 0
    def zero_grad(s):
        for p in s.ep + s.tp: p.grad = None
    def reset(s, k):
        for p in s.ep: s.m[id(p)][k] = 0; s.v[id(p)][k] = 0
        s.t_rows[k] = 0
    @torch.no_grad()
    def step(s, lr, fac=None, fac_s=1.0):
        if fac is not None and s.ep:
            act = fac > 0
            if act.any():
                s.t_rows[act] += 1; bc1, bc2 = 1 - s.b1 ** s.t_rows[act], 1 - s.b2 ** s.t_rows[act]
                for p in s.ep:
                    if p.grad is None: continue
                    g = p.grad[act]; m, v = s.m[id(p)], s.v[id(p)]
                    m[act] = s.b1 * m[act] + (1 - s.b1) * g; v[act] = s.b2 * v[act] + (1 - s.b2) * g * g
                    sh = (-1,) + (1,) * (p.dim() - 1)
                    p[act] -= lr * fac[act].view(sh) * (m[act] / bc1.view(sh)) / ((v[act] / bc2.view(sh)).sqrt() + s.eps)
        if fac_s > 0 and s.tp:
            s.t_trunk += 1; bc1, bc2 = 1 - s.b1 ** s.t_trunk, 1 - s.b2 ** s.t_trunk
            for p in s.tp:
                if p.grad is None: continue
                m, v = s.m[id(p)], s.v[id(p)]
                m.mul_(s.b1).add_(p.grad, alpha=1 - s.b1); v.mul_(s.b2).addcmul_(p.grad, p.grad, value=1 - s.b2)
                p.add_((m / bc1) / ((v / bc2).sqrt() + s.eps), alpha=-lr * fac_s)

class LossScaler:
    def __init__(s, enabled, init=2.0 ** 14, interval=1000):
        s.enabled, s.scale, s.interval, s.good = enabled, (init if enabled else 1.0), interval, 0
    def unscale_and_check(s, params):
        if not s.enabled: return True
        inv, finite = 1.0 / s.scale, True
        for p in params:
            if p.grad is not None:
                p.grad.mul_(inv)
                if finite and not torch.isfinite(p.grad).all(): finite = False
        if not finite: s.scale /= 2; s.good = 0; return False
        s.good += 1
        if s.good % s.interval == 0: s.scale *= 2
        return True

## 5. Learners
* **Population** (U and its variants, BAL, Oracle-experts). Per batch: E-step on the first 128 tokens; creation when more than 65% of the batch is uncovered by every fitted component; reaction (U: mass += assigned count; BAL: renormalised, horizon 84 batches); transport factors min(1, A·b_c/m_c) and min(1, A·b/N_fit). **U-K4** holds at most 4 components: when a new one is needed and the pool is full, the two most similar fitted components (cosine of their weight updates) are merged into their mass-weighted update, re-factorised to rank 16, and their masses and evidence are added. This is the WFR transport of two masses to one point; it needs no data. **U-nocreate** has all 12 components from the start. **Oracle-experts** assigns by the true domain label.
* **LoRA-match** and its options: `schedule` (U's shared-adapter schedule, the frontier control), `lr_scale`, `oewc` (online EWC, consolidated every 100 steps, decay 0.9, Fisher normalised to mean 1), `replay` (reservoir of 2% of the stream, 4 of 16 sequences per batch replayed after the first segment).
* **Full-FT:** all 160M weights, AdamW. **MoLoRA-router:** U's adapters, all present, mixed per token by a top-2 router with the Switch load-balancing loss.

In [ ]:
class LoRAMatch:
    """Single LoRA on all layers, same trainable parameters as U at K_max. Options:
       schedule: U's shared-adapter step schedule (frontier control); lr_scale: learning-rate frontier;
       oewc: online EWC, consolidated every oewc_every steps (task-free); replay: ER reservoir (stores data)."""
    particles = False; oracle = False
    def __init__(s, spec):
        s.spec = spec; s.model = AdaptedLM(MATCH_RANK); s.opt = ParticleAdamW(s.model); s.scaler = LossScaler(AMP)
        s.it, s.log, s.skipped = 0, [], 0; s.lr_scale = spec.get("lr_scale", 1.0); s.lr_base = CFG["lr_lora"]
        if spec.get("oewc"): s.F = None; s.anchor = None; s.Fwin = [torch.zeros_like(p) for p in s.opt.tp]; s.nwin = 0
        if spec.get("replay"):
            total = CFG["steps_per_segment"] * len(TEST) * CFG["batch"]; s.cap = max(CFG["batch"], int(CFG["er_fraction"] * total))
            s.buf_x, s.buf_y, s.seen = [], [], 0; s.rgen = torch.Generator().manual_seed(77)
    def params(s): return s.opt.ep + s.opt.tp
    def _update(s, loss, lr, fac=None, fac_s=1.0):
        s.opt.zero_grad(); (loss * s.scaler.scale).backward()
        if not s.scaler.unscale_and_check(s.params()): s.skipped += 1; return False
        torch.nn.utils.clip_grad_norm_(s.params(), CFG["clip"]); s.opt.step(lr, fac, fac_s); return True
    def _replay(s, x, y):
        for i in range(x.shape[0]):
            s.seen += 1
            if len(s.buf_x) < s.cap: s.buf_x.append(x[i].cpu()); s.buf_y.append(y[i].cpu())
            else:
                j = int(torch.randint(s.seen, (1,), generator=s.rgen))
                if j < s.cap: s.buf_x[j] = x[i].cpu(); s.buf_y[j] = y[i].cpu()
        if s.it > CFG["steps_per_segment"]:
            n = CFG["er_replay"]; idx = torch.randint(len(s.buf_x), (n,), generator=s.rgen)
            x = torch.cat([x[:-n], torch.stack([s.buf_x[i] for i in idx]).to(DEVICE)]); y = torch.cat([y[:-n], torch.stack([s.buf_y[i] for i in idx]).to(DEVICE)])
        return x, y
    def step(s, x, y, lr_mult, doms=None):
        s.it += 1
        if s.spec.get("replay"): x, y = s._replay(x, y)
        with autocast(): loss = s.model.seq_nll(x, y).mean()
        total = loss
        if s.spec.get("oewc") and s.F is not None:
            total = total + 0.5 * CFG["oewc_lambda"] * sum((f * (p - a) ** 2).sum() for f, p, a in zip(s.F, s.opt.tp, s.anchor))
        fac_s = 1.0
        if s.spec.get("schedule") and s.it >= CFG["E_min"]: fac_s = min(1.0, CFG["A_s"] / s.it)
        ok = s._update(total, s.lr_base * s.lr_scale * lr_mult, None, fac_s)
        if s.spec.get("oewc") and ok:
            with torch.no_grad():
                for w, p, i in zip(s.Fwin, s.opt.tp, range(len(s.Fwin))):
                    g = p.grad if s.F is None else p.grad - CFG["oewc_lambda"] * s.F[i] * (p - s.anchor[i])
                    w += g.float() ** 2
                s.nwin += 1
                if s.it % CFG["oewc_every"] == 0:
                    Fn = [w / s.nwin for w in s.Fwin]; mean = torch.cat([f.flatten() for f in Fn]).mean().clamp_min(1e-20); Fn = [f / mean for f in Fn]
                    s.F = Fn if s.F is None else [CFG["oewc_gamma"] * a + b for a, b in zip(s.F, Fn)]
                    s.anchor = [p.detach().clone() for p in s.opt.tp]; s.Fwin = [torch.zeros_like(p) for p in s.opt.tp]; s.nwin = 0
        return float(loss.detach())
    @torch.no_grad()
    def seq_losses(s, x, y):
        CTX.aux = []
        with autocast(): return s.model.seq_nll(x, y).float()
    def exist(s): return [None]

class FullFT(LoRAMatch):
    """full fine-tuning of all pretrained weights (common practice); torch AdamW + GradScaler"""
    def __init__(s, spec):
        s.spec = spec; s.model = AdaptedLM(0); s.it, s.log, s.skipped = 0, [], 0
        for p in s.model.lm.parameters(): p.requires_grad_(True)
        s.o = torch.optim.AdamW(s.model.lm.parameters(), lr=CFG["lr_full"], weight_decay=0.0); s.sc = torch.amp.GradScaler("cuda", enabled=AMP)
    def step(s, x, y, lr_mult, doms=None):
        s.it += 1
        for gr in s.o.param_groups: gr["lr"] = CFG["lr_full"] * lr_mult
        with autocast(): loss = s.model.seq_nll(x, y).mean()
        s.o.zero_grad(); s.sc.scale(loss).backward(); s.sc.unscale_(s.o)
        torch.nn.utils.clip_grad_norm_(s.model.lm.parameters(), CFG["clip"]); scale = s.sc.get_scale(); s.sc.step(s.o); s.sc.update()
        if s.sc.get_scale() < scale: s.skipped += 1
        return float(loss.detach())
    def n_trainable(s): return sum(p.numel() for p in s.model.lm.parameters())

class MoLoRARouter(LoRAMatch):
    """U's shared adapter and K components, all present from the start, mixed per token by a learned top-2 router"""
    def __init__(s, spec):
        s.spec = spec; s.model = AdaptedLM(CFG["shared_rank"], CFG["K_max"], CFG["comp_rank"], CFG["comp_layers"], CFG["router_topk"])
        s.opt = ParticleAdamW(s.model); s.scaler = LossScaler(AMP); s.it, s.log, s.skipped = 0, [], 0
        s.ones = torch.ones(CFG["K_max"], device=DEVICE)
    def step(s, x, y, lr_mult, doms=None):
        s.it += 1; CTX.aux = []
        with autocast():
            loss = s.model.seq_nll(x, y).mean(); total = loss + CFG["aux_coef"] * sum(CTX.aux) / max(1, len(CTX.aux))
        s._update(total, CFG["lr_lora"] * lr_mult, s.ones, 1.0); return float(loss.detach())

class Population(LoRAMatch):
    """Unbalanced WFR (U) and its variants.
       E-step    gamma_c(x) ∝ m_c rho_c(x) on the first `prefix` tokens (exact Bayes)
       Reaction  m_c += sequences assigned (U); BAL renormalises its masses. Promotion counts raw evidence in both.
       Transport component min(1, A_p b_c/m_c); shared adapter min(1, A_s b/N_fit); newborn free. BAL / trunk_free: shared 1.
       Creation  a region (> nu of the batch) uncovered by every fitted component goes to the newborn (cloned from the closest).
       budget    at most K_budget components; when full, the two most similar are merged (mass-weighted, data-free)
       nocreate  all K_max components from the start, no creation
       oracle    one component per TRUE domain (labels!): an upper reference for creation and routing"""
    particles = True
    def __init__(s, spec):
        s.spec = spec; s.balanced = spec.get("balanced", False); s.oracle = spec.get("oracle", False)
        K = CFG["K_budget"] if spec.get("budget") else CFG["K_max"]; s.K = K
        s.model = AdaptedLM(CFG["shared_rank"], K, CFG["comp_rank"], CFG["comp_layers"])
        s.opt = ParticleAdamW(s.model); s.scaler = LossScaler(AMP); s.it, s.log, s.skipped = 0, [], 0
        z = lambda dt=torch.float32: torch.zeros(K, device=DEVICE, dtype=dt)
        s.m, s.ev = z(), z(); s.is_ex, s.promoted, s.fresh = z(torch.bool), z(torch.bool), z(torch.bool)
        s.level = torch.full((K,), float("nan"), device=DEVICE); s.trace = []; s.dom2comp = {}
        s.free = 0; s.is_ex[0] = True; s.m[0] = 1e-3; s.fresh[0] = True
        if spec.get("nocreate"): s.is_ex[:] = True; s.promoted[:] = True; s.m[:] = 1e-3; s.free = -1
        if s.oracle: s.free = -1
    def exist(s): return s.is_ex.nonzero().squeeze(1).tolist()
    def mass(s): w = s.m * s.is_ex; return w / w.sum()
    def _merge_closest(s):
        pr = [c for c in s.exist() if s.promoted[c]]
        D = {c: torch.cat([d.flatten() for d in s.model.deltas(c)]) for c in pr}; best, pair = -2.0, None
        for i in range(len(pr)):
            for j in range(i + 1, len(pr)):
                cs = float(F.cosine_similarity(D[pr[i]], D[pr[j]], dim=0))
                if cs > best: best, pair = cs, (pr[i], pr[j])
        i, j = pair; wi, wj = float(s.m[i]), float(s.m[j])
        merged = [(wi * a + wj * b) / (wi + wj) for a, b in zip(s.model.deltas(i), s.model.deltas(j))]
        s.model.set_from_deltas(i, merged, CFG["comp_rank"])
        if not torch.isnan(s.level[j]): s.level[i] = (wi * s.level[i] + wj * s.level[j]) / (wi + wj)
        s.m[i] += s.m[j]; s.ev[i] += s.ev[j]; s.m[j] = 0; s.ev[j] = 0; s.is_ex[j] = False; s.promoted[j] = False
        s.opt.reset(i); s.opt.reset(j); s.log.append([s.it, "merge", int(j), int(i), round(best, 3)]); return i, j
    def _spawn(s, parent):
        free = (~s.is_ex).nonzero().squeeze(1)
        if len(free) == 0:
            if s.spec.get("budget"):
                i, j = s._merge_closest(); parent = i if parent == j else parent; free = (~s.is_ex).nonzero().squeeze(1)
            else: s.free = -1; s.log.append([s.it, "pool full"]); return
        k = int(free[0]); s.model.clone(parent, k); s.opt.reset(k)
        s.is_ex[k] = True; s.m[k] = 1e-3; s.ev[k] = 0; s.level[k] = float("nan"); s.fresh[k] = True; s.free = k
        s.log.append([s.it, "birth", k, int(parent)])
    def _promote(s, why):
        s.promoted[s.free] = True; s.log.append([s.it, "promote", s.free, why]); s._spawn(s.free)
    @torch.no_grad()
    def nll_matrix(s, x, y, parts, P=None):
        P = P or x.shape[1]; xp, yp = x[:, :P], y[:, :P]; B = x.shape[0]
        if s.model.split_ok is None: s.model.check_split(x[:2, :64], y[:2, :64])
        chunk = max(1, (CFG["batch"] * CFG["seq_len"]) // (B * P))
        with autocast(): return s.model.nll_parts(xp, yp, parts, chunk).float()
    def _oracle_assign(s, x, y, doms):
        for d in dict.fromkeys(doms):
            if d not in s.dom2comp:
                free = (~s.is_ex).nonzero().squeeze(1)
                k = 0 if not s.dom2comp else int(free[0])
                if s.dom2comp:   # born at the closest existing component
                    ex = s.exist(); sel = torch.tensor([dd == d for dd in doms], device=DEVICE)
                    nl = s.nll_matrix(x[sel], y[sel], ex, CFG["prefix"]); par = ex[int(nl.mean(1).argmin())]
                    s.model.clone(par, k); s.opt.reset(k); s.log.append([s.it, "create", k, d])
                s.is_ex[k] = True; s.promoted[k] = True; s.m[k] = max(float(s.m[k]), 1e-3); s.dom2comp[d] = k
        return torch.tensor([s.dom2comp[d] for d in doms], device=DEVICE)
    def step(s, x, y, lr_mult, doms=None):
        s.it += 1; b = x.shape[0]; P = CFG["prefix"]; unc = torch.zeros(b, dtype=torch.bool, device=DEVICE); ex = s.exist()
        if s.oracle: assign = s._oracle_assign(x, y, doms)
        else:
            nll = s.nll_matrix(x, y, ex, P)
            if not s.spec.get("nocreate"):
                lev = s.level[ex][:, None]
                cov = (nll <= lev + CFG["delta"]) & ~torch.isnan(lev) & s.promoted[ex][:, None]
                unc = ~cov.any(0); region = float(unc.float().mean()) > CFG["nu_region"]
            else: region = False
            nov = unc & region
            if region and s.free >= 0:                                     # ---- creation ----
                fi = ex.index(s.free)
                fc = float((nll[fi, nov] <= s.level[s.free] + CFG["delta"]).float().mean()) if not torch.isnan(s.level[s.free]) else 0.0
                if s.ev[s.free] >= CFG["E_floor"] * b and fc < 0.5: s._promote("new region")
                if s.free >= 0 and s.ev[s.free] < b:
                    parent = ex[int(nll[:, nov].mean(1).argmin())]
                    if parent != s.free: s.model.clone(parent, s.free); s.opt.reset(s.free); s.log.append([s.it, "reborn", s.free, parent])
                old = {c: nll[i] for i, c in enumerate(ex) if c != s.free and s.is_ex[c]}
                ex = s.exist(); new = [c for c in ex if c not in old]
                if new:
                    rows = dict(zip(new, s.nll_matrix(x, y, new, P))); nll = torch.stack([old[c] if c in old else rows[c] for c in ex])
                else: nll = torch.stack([old[c] for c in ex])
            logit = s.m[ex].log()[:, None] - nll * P                        # ---- E-step ----
            assign = torch.tensor(ex, device=DEVICE)[torch.distributions.Categorical(logits=logit.T).sample()]
            if s.free >= 0 and nov.any():
                assign[nov] = s.free
                if s.fresh[s.free]: s.log.append([s.it, "create", s.free, int(nov.sum())]); s.fresh[s.free] = False
            for i, c in enumerate(ex):                                      # running level on each component's own data
                sel = assign == c
                if sel.sum() >= 2:
                    q = float(torch.quantile(nll[i, sel], CFG["level_q"]))
                    s.level[c] = q if torch.isnan(s.level[c]) else (1 - CFG["level_rate"]) * s.level[c] + CFG["level_rate"] * q
        cnt = torch.bincount(assign, minlength=s.K).float()                 # ---- reaction ----
        s.m = (s.m * (1 - 1 / CFG["bal_window"]) * s.is_ex + cnt) if s.balanced else (s.m + cnt); s.ev += cnt
        CTX.gate = F.one_hot(assign, s.K).float()                           # ---- M-step (transport) ----
        with autocast(): loss = s.model.seq_nll(x, y).mean()
        if s.balanced: fac, fac_s = (cnt > 0).float(), 1.0
        else:
            fac = torch.clamp(CFG["A_p"] * cnt / s.m.clamp(min=1), max=1.0)
            n_fit = float(s.m[s.promoted].sum())
            fac_s = 1.0 if (s.spec.get("trunk_free") or n_fit == 0) else min(1.0, CFG["A_s"] * b / n_fit)
        if s.free >= 0 and cnt[s.free] > 0: fac[s.free] = 1.0
        s._update(loss, CFG["lr_lora"] * lr_mult, fac, fac_s)
        if s.free >= 0 and s.ev[s.free] >= CFG["E_min"] * b: s._promote("evidence")
        if s.it % 25 == 0: s.trace.append([s.it, round(float(unc.float().mean()), 3), round(fac_s, 4), len(s.exist())])
        return float(loss.detach())
    @torch.no_grad()
    def seq_losses(s, x, y, return_post=False):
        ex = s.exist(); T = x.shape[1]; nll = s.nll_matrix(x, y, ex)
        lj = s.mass()[ex].clamp_min(1e-30).log()[:, None] - nll * T; mix = -torch.logsumexp(lj, 0) / T
        return (mix, lj.softmax(0), ex) if return_post else mix

def make_learner(name):
    spec = CONDITIONS[name]; k = spec["kind"]
    return {"lora": LoRAMatch, "full": FullFT, "moe": MoLoRARouter, "pop": Population}[k](spec)

## 6. Evaluation
Held-out loss per domain (nats per token, 24 windows of 512 tokens). Populations are scored as the sequence-level mixture Σ π_c ρ_c(x), π = m/Σm, with no label. Primary metrics use the first 12 segments; the 13th (recurrence) is reported separately. General ability: LAMBADA last-word accuracy (500 examples) and WikiText-103 test loss, at the start and end.

In [ ]:
@torch.no_grad()
def evaluate_domains(L, domains):
    L.model.eval(); out, post, labels, routes = {}, {}, [], []
    for di, t in enumerate(domains):
        x, y = EVAL[t]; vals, P = [], []
        for i in range(0, len(x), 8):
            if L.particles:
                mix, pst, ex = L.seq_losses(x[i:i + 8], y[i:i + 8], True); vals.append(mix); P.append(pst)
                routes += [ex[j] for j in pst.argmax(0).tolist()]; labels += [di] * len(mix)
            else: vals.append(L.seq_losses(x[i:i + 8], y[i:i + 8]))
        out[t] = float(torch.cat(vals).mean())
        if L.particles: post[t] = torch.cat(P, 1).mean(1).tolist()
    L.model.train(); res = dict(loss=out)
    if L.particles:
        from sklearn.metrics import adjusted_rand_score
        res.update(post=post, ex=L.exist(), mass=L.mass().tolist(), ari=float(adjusted_rand_score(labels, routes)))
    return res

GENERAL = None
def load_general():
    global GENERAL
    if GENERAL is not None or OFFLINE: return GENERAL
    from datasets import load_dataset
    lam = load_dataset("EleutherAI/lambada_openai", "default", split="test")["text"][:CFG["lambada_n"]]; ex = []
    for t in lam:
        ctx, w = t.rsplit(" ", 1); ex.append((TOK(ctx)["input_ids"], TOK(" " + w)["input_ids"]))
    wt = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")["text"]
    ids = torch.tensor(TOK(["".join(wt[:4000])])["input_ids"][0], dtype=torch.long)
    GENERAL = dict(lambada=ex, wiki=eval_windows(ids, CFG["wikitext_windows"], CFG["seq_len"])); return GENERAL

@torch.no_grad()
def evaluate_general(L):
    G = load_general()
    if G is None: return {}
    L.model.eval(); xw, yw = G["wiki"]
    wl = torch.cat([L.seq_losses(xw[i:i + 8], yw[i:i + 8]) for i in range(0, len(xw), 8)]).mean(); correct = 0
    for i in range(0, len(G["lambada"]), 16):
        batch = G["lambada"][i:i + 16]; n = max(len(c) + len(g) for c, g in batch)
        X = torch.full((len(batch), n), TOK.eos_token_id, dtype=torch.long)
        for j, (c, g) in enumerate(batch): X[j, :len(c) + len(g)] = torch.tensor(c + g)
        X = X.to(DEVICE); best = [float("inf")] * len(batch); okb = [False] * len(batch)
        for p in L.exist():
            if p is not None: CTX.gate = F.one_hot(torch.full((len(batch),), p, device=DEVICE), L.K).float()
            CTX.aux = []
            with autocast(): lp = L.model.lm(input_ids=X).logits.float().log_softmax(-1)
            for j, (c, g) in enumerate(batch):
                cn = float(-lp[j, torch.arange(len(c) - 1, device=DEVICE), torch.tensor(c[1:], device=DEVICE)].sum())
                ok = bool((lp[j, len(c) - 1:len(c) + len(g) - 1].argmax(-1).cpu() == torch.tensor(g)).all())
                if cn < best[j]: best[j], okb[j] = cn, ok
        correct += sum(okb)
    L.model.train(); return dict(wikitext_loss=float(wl), lambada_acc=correct / len(G["lambada"]))

def summarize(M, n_unique):
    """M rows: after each segment (row 0 = before training); columns: unique domains in learning order.
    Primary metrics use the first n_unique segments (before the recurrence)."""
    M = np.array(M); T = n_unique
    out = dict(final=float(M[T].mean()), forgetting=float(np.mean([M[T, t] - M[t + 1, t] for t in range(T - 1)])),
               plasticity=float(np.mean([M[t + 1, t] for t in range(T)])))
    out["switch_forgetting"] = [float(np.mean([M[s + 1, t] - M[s, t] for t in range(s)])) for s in range(1, T)]
    if M.shape[0] > T + 1: out["recurrence"] = dict(before=float(M[T, 0]), first=float(M[1, 0]), after=float(M[T + 1, 0]),
                                                   final_all=float(M[T + 1].mean()))
    return out

## 7. Training loop, storage and run control
`run_condition(name)` runs the tier's seeds and skips saved ones. `run_tier(t)` runs a whole tier, seeds outer. `budget()` shows the GPU time used and the runs left.

In [ ]:
RESULTS_PATH = os.path.join(ROOT, "results.json")
def load_results(path=RESULTS_PATH): return json.load(open(path)) if os.path.exists(path) else {}
def cfg_key(): return {k: v for k, v in CFG.items() if not k.startswith("cal_")}

def train_one(name, seed, which="test", steps=None, general=True):
    spec = CONDITIONS[name]; segs = make_stream(seed, which); S = steps or CFG["steps_per_segment"]
    uniq = list(dict.fromkeys(segs)); mode = "joint" if spec.get("joint") else "gradual" if spec.get("gradual") else "hard"
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed); gen = torch.Generator().manual_seed(seed)
    L = make_learner(name); print(f"  trainable parameters: {L.model.n_trainable() / 1e6:.2f}M" if not isinstance(L, FullFT) else f"  trainable parameters: {L.n_trainable() / 1e6:.1f}M (full)")
    ev = evaluate_domains(L, uniq); M = [[ev["loss"][t] for t in uniq]]; g0 = evaluate_general(L) if general else {}
    stages, losses, hashes, t0 = [], [], [], time.time()
    for step in range(S * len(segs)):
        x, y, doms = batch_for(segs, step, S, gen, mode)
        if step % S == 0: hashes.append(hashlib.md5(x.cpu().numpy().tobytes()).hexdigest()[:12])
        lr_mult = min(1.0, (step + 1) / CFG["warmup"])
        losses.append(L.step(x, y, lr_mult, doms if L.oracle else None))    # only the oracle reference sees labels
        if step == 49:
            per = (time.time() - t0) / 50; print(f"  [{name} s{seed}] {per * 1000:.0f} ms/step -> about {per * S * len(segs) / 60:.0f} min of training")
        if (step + 1) % S == 0:
            s = (step + 1) // S; ev = evaluate_domains(L, uniq); M.append([ev["loss"][t] for t in uniq]); stages.append(ev)
            extra = f" | parts {len(ev['ex'])}, ARI {ev['ari']:.2f}" if L.particles else ""
            print(f"  [{name} s{seed}] seg {s:2d} ({segs[s - 1][:18]:18s}) mean loss {np.mean(M[-1]):.3f}  ({(time.time() - t0) / 60:.1f} min){extra}")
    g1 = evaluate_general(L) if general else {}
    res = dict(M=M, order=uniq, segments=segs, minutes=(time.time() - t0) / 60, skipped=L.skipped, steps=S * len(segs),
               losses=losses[::5], batch_hashes=hashes, stages=stages, events=L.log, trace=getattr(L, "trace", []),
               general0=g0, general1=g1, **summarize(M, len(uniq)))
    del L
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return res

def run_condition(name, seeds=None):
    if not os.path.exists(FROZEN_PATH) and not ALLOW_UNFROZEN: load_frozen(True)
    load_frozen(required=not ALLOW_UNFROZEN)
    R = load_results()
    if "_cfg" in R and R["_cfg"] != json.loads(json.dumps(cfg_key())):
        raise RuntimeError("results.json was made with different settings; use a new folder (change ROOT).")
    for seed in (seeds if seeds is not None else SEEDS[CONDITIONS[name]["tier"]]):
        if str(seed) in R.get(name, {}): print(f"{name} seed {seed}: already done, skipping"); continue
        print(f"=== {name} | seed {seed} | segments: {' -> '.join(make_stream(seed))} ===")
        res = train_one(name, seed)
        R = load_results(); R["_cfg"] = cfg_key(); R.setdefault(name, {})[str(seed)] = res; json.dump(R, open(RESULTS_PATH, "w"))
        print(f"  final {res['final']:.4f} | forgetting {res['forgetting']:+.4f} | plasticity {res['plasticity']:.4f} nats/token ({res['minutes']:.0f} min)")

def run_single_domain(seed=0):
    """fresh-plasticity reference: LoRA-match trained on each test domain alone, from the base model"""
    load_frozen(required=not ALLOW_UNFROZEN); R = load_results(); key = f"Single-domain"
    if str(seed) in R.get(key, {}): print("Single-domain: already done"); return
    out = {}
    for d in TEST:
        gen = torch.Generator().manual_seed(seed); L = make_learner("LoRA-match")
        for step in range(CFG["steps_per_segment"]):
            x, y, _ = batch_for([d], step, CFG["steps_per_segment"], gen); L.step(x, y, min(1.0, (step + 1) / CFG["warmup"]))
        out[d] = evaluate_domains(L, [d])["loss"][d]; print(f"  {d:20s} {out[d]:.4f}"); del L
    R = load_results(); R.setdefault(key, {})[str(seed)] = out; json.dump(R, open(RESULTS_PATH, "w"))
print("ready. saved so far:", {k: sorted(v) for k, v in load_results().items() if k != "_cfg"})

def run_tier(tier, conditions=None):
    """seeds outer, conditions inner: if the budget runs out, every condition has the same seeds"""
    names = conditions or [n for n, c in CONDITIONS.items() if c["tier"] == tier]
    for seed in SEEDS[tier]:
        for n in names: run_condition(n, [seed])

def budget():
    R = load_results(); used = sum(r["minutes"] for k, v in R.items() if k != "_cfg" for r in v.values() if isinstance(r, dict) and "minutes" in r)
    C = load_results(os.path.join(ROOT, "tier0.json")); used += sum(v.get("minutes", 0) for v in C.values())
    print(f"GPU time used so far: {used / 60:.1f} h (training and evaluation; download and model loading not included)")
    left = [(n, s) for n, c in CONDITIONS.items() for s in SEEDS[c["tier"]] if str(s) not in R.get(n, {})]
    print(f"runs left: {len(left)}" + (": " + ", ".join(f"{n} s{s}" for n, s in left[:12]) + (" ..." if len(left) > 12 else "") if left else ""))

## 8. Calibration and pre-registration code

In [ ]:
CAL_PATH = os.path.join(ROOT, "tier0.json")
CODE_OBJECTS = ["AdaptedLinear", "AdaptedLM", "ParticleAdamW", "LoRAMatch", "FullFT", "MoLoRARouter", "Population",
                "batch_for", "make_stream", "evaluate_domains", "evaluate_general", "summarize", "train_one", "report"]
def code_hash():
    """hash of the method and evaluation code (inspect works for notebook cells as well)"""
    import inspect; h = hashlib.sha256()
    for n in CODE_OBJECTS:
        try: h.update(inspect.getsource(globals()[n]).encode())
        except (OSError, TypeError, KeyError): h.update(n.encode())
    return h.hexdigest()[:16]

def prereg():
    """everything the pre-registration must fix, and its hash. File the printout (e.g. on OSF) before Tier 1."""
    fz = load_frozen(required=not ALLOW_UNFROZEN)
    rec = dict(cfg=cfg_key(), frozen={k: CFG[k] for k in TUNED}, conditions=CONDITIONS, seeds=SEEDS,
               streams={s: make_stream(s) for s in range(3)}, match_rank=MATCH_RANK, code_hash=code_hash(),
               hypotheses=dict(H1="forgetting(U) <= 0.7 forgetting(BAL), paired over seeds",
                               H2a="final(U) < final(LoRA-match)", H2b="final(U) < final(MoLoRA-router)",
                               H3="U below the LoRA forgetting-plasticity frontier at U's plasticity",
                               H4="U-K4 keeps >= 70% of U's final-loss gain over LoRA-match",
                               H5="U's LAMBADA drop and WikiText rise no worse than LoRA-match's",
                               test="paired t over seeds, Holm over H1, H2a, H2b at 0.05; switch-level cluster bootstrap for forgetting"))
    txt = json.dumps(rec, sort_keys=True, default=str); digest = hashlib.sha256(txt.encode()).hexdigest()
    json.dump(dict(rec, sha256=digest), open(os.path.join(ROOT, "preregistration.json"), "w"), indent=1, default=str)
    print(json.dumps(rec, indent=1, default=str)[:4000]); print("\nPRE-REGISTRATION SHA-256:", digest)
    return digest
def calibrate():
    """Tier 0 on the development stream (pre-registered rule: lowest final mean loss), then freeze the settings."""
    C = load_results(CAL_PATH); base = {k: CFG[k] for k in TUNED}
    def run(name, key, overrides):
        if key in C: return C[key]["final"]
        overrides = dict(scaled(CFG["cal_steps"]), **overrides)      # U's step-based settings scaled to the shorter segments
        old = {k: CFG[k] for k in overrides}; CFG.update(overrides)
        print(f"--- Tier 0: {name} {overrides}")
        r = train_one(name, 0, which="dev", steps=CFG["cal_steps"], general=False); CFG.update(old)
        C[key] = dict(final=r["final"], forgetting=r["forgetting"], plasticity=r["plasticity"], overrides=overrides, minutes=r["minutes"])
        json.dump(C, open(CAL_PATH, "w")); print(f"    final {r['final']:.4f}"); return r["final"]
    best = {}
    sc = {lr: run("LoRA-match", f"lora_lr={lr}", {"lr_lora": lr}) for lr in CFG["cal_lr_lora"]}; best["lr_lora"] = min(sc, key=sc.get)
    sc = {lr: run("Full-FT", f"full_lr={lr}", {"lr_full": lr}) for lr in CFG["cal_lr_full"]}; best["lr_full"] = min(sc, key=sc.get)
    sc = {d: run("U", f"U_delta={d}", {"delta": d, "lr_lora": best["lr_lora"]}) for d in CFG["cal_delta"]}; best["delta"] = min(sc, key=sc.get)
    sc = {l: run("OEWC", f"oewc={l}", {"oewc_lambda": l, "lr_lora": best["lr_lora"]}) for l in CFG["cal_oewc"]}; best["oewc_lambda"] = min(sc, key=sc.get)
    frozen = dict(best, frozen_at=time.strftime("%Y-%m-%d %H:%M:%S"), rule="lowest final mean loss on the development stream",
                  tier0=C, code_hash=code_hash())
    json.dump(frozen, open(FROZEN_PATH, "w"), indent=1); CFG.update(best)
    print("\nFROZEN SETTINGS:", best, "\nFile:", FROZEN_PATH, "\nFile this with the pre-registration before running Tier 1.")
    return frozen

## 9. Tier 0: calibration on the development stream
Selection rule (fixed in the plan): for each tuned value, the setting with the lowest final mean loss on the development stream, 5 domains × 150 steps, seed 0 (U's step-based settings scaled to 150-step segments). LoRA's learning rate is chosen first and is shared by every LoRA-based method (U, BAL, MoLoRA, OEWC, ER, Joint). U's δ is shared by BAL and every U variant. Runs are saved one by one in `tier0.json`. About 3 hours.

In [ ]:
FROZEN = calibrate()

## 10. Pre-registration
Prints the settings, conditions, seeds, the three seed streams, the hypotheses and tests, a hash of the method code, and the SHA-256 of all of it, and saves `preregistration.json`. **File it (e.g. on OSF) before running Tier 1**, so the results can be checked against a plan fixed in advance.

In [ ]:
load_frozen()
PREREG_HASH = prereg()

## 11. Tier 1: primary comparison (3 seeds)
21 runs. The first line of each run prints its measured speed and the expected minutes.

In [ ]:
run_tier(1)

Quick look after Tier 1 (optional): `report()`

## 12. Tier 2: bounded size and ablations (3 seeds)

In [ ]:
run_tier(2)

## 13. Tier 3: references (2 seeds) and the fresh-model reference (1 seed)
Oracle-experts sees domain labels and ER-2% stores data; both are references, not competitors. Single-domain trains a fresh LoRA-match on each test domain alone, to measure loss of plasticity.

In [ ]:
run_tier(3)
run_single_domain(0)

## 14. Tier 4: gradual stream (1 seed)
At each boundary, the 60 steps around it (20% of a segment) blend the two domains per sequence, linearly from 0% to 100%.

In [ ]:
run_tier(4)

## 15. Budget

In [ ]:
budget()

## 16. Results
Table (mean ± 95% CI over seeds), the pre-registered tests with Holm correction, H3–H5, creation and routing statistics, and plots: forgetting vs. plasticity, routing heatmaps, creation timelines, plasticity vs. fresh models.

In [ ]:
from math import sqrt
def _tci(v):
    from scipy import stats
    v = np.asarray(v, float); n = len(v); sd = v.std(ddof=1) if n > 1 else 0.0
    h = stats.t.ppf(0.975, n - 1) * sd / sqrt(n) if n > 1 else float("nan"); return v.mean(), sd, h

def _paired(R, a, b, metric):
    from scipy import stats
    common = sorted(set(R.get(a, {})) & set(R.get(b, {})), key=int)
    if not common: return None
    va = np.array([R[a][s][metric] for s in common]); vb = np.array([R[b][s][metric] for s in common]); d = va - vb
    mu, sd, h = _tci(d); p = stats.ttest_rel(va, vb).pvalue if len(common) > 1 else float("nan")
    # switch-level cluster bootstrap (forgetting only): resample seeds, then switches within seed
    boot = None
    if metric == "forgetting":
        sw = [np.array(R[a][s]["switch_forgetting"]) - np.array(R[b][s]["switch_forgetting"]) for s in common]
        rng = np.random.default_rng(0); bs = []
        for _ in range(2000):
            ss = [sw[i] for i in rng.integers(len(sw), size=len(sw))]; bs.append(np.mean([x[rng.integers(len(x), size=len(x))].mean() for x in ss]))
        boot = (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))
    return dict(n=len(common), diff=mu, ci=h, p=p, boot=boot, a=float(va.mean()), b=float(vb.mean()))

def report():
    R = load_results(); names = [n for n in CONDITIONS if n in R]
    print("nats per token; final, forgetting, plasticity: lower is better\n")
    print(f"{'condition':20s} {'n':>2s} {'final':>16s} {'forgetting':>16s} {'plasticity':>16s} {'LAMBADA drop':>13s} {'Wiki rise':>10s} {'min':>5s}")
    for n in names:
        sd = sorted(R[n], key=int); row = [_tci([R[n][s][m] for s in sd]) for m in ("final", "forgetting", "plasticity")]
        lam = np.mean([R[n][s]["general0"].get("lambada_acc", np.nan) - R[n][s]["general1"].get("lambada_acc", np.nan) for s in sd])
        wik = np.mean([R[n][s]["general1"].get("wikitext_loss", np.nan) - R[n][s]["general0"].get("wikitext_loss", np.nan) for s in sd])
        print(f"{n:20s} {len(sd):2d} " + " ".join(f"{mu:8.4f} ±{h:6.4f}" for mu, _, h in row) + f" {lam:13.3f} {wik:10.3f} {np.mean([R[n][s]['minutes'] for s in sd]):5.0f}")
    # primary, Holm-corrected
    prim = [("H1", "U", "BAL", "forgetting"), ("H2a", "U", "LoRA-match", "final"), ("H2b", "U", "MoLoRA-router", "final")]
    rows = [(h, a, b, m, _paired(R, a, b, m)) for h, a, b, m in prim]; valid = sorted([r for r in rows if r[4] and not np.isnan(r[4]["p"])], key=lambda r: r[4]["p"])
    adj, run = {}, 0.0
    for k, r in enumerate(valid): run = max(run, min(1.0, (len(valid) - k) * r[4]["p"])); adj[r[0]] = run
    print("\nPRIMARY (pre-registered; first minus second; negative = U better)")
    for h, a, b, m, r in rows:
        if not r: print(f"  {h}: missing runs"); continue
        extra = f", forgetting ratio {r['a'] / r['b']:.2f} (H1 needs <= 0.70)" if h == "H1" and r["b"] > 0 else ""
        bs = f", switch-level bootstrap [{r['boot'][0]:+.4f}, {r['boot'][1]:+.4f}]" if r["boot"] else ""
        print(f"  {h} {a} - {b} {m}: {r['diff']:+.4f} ± {r['ci']:.4f} (n={r['n']}), p={r['p']:.3g}, Holm p={adj.get(h, float('nan')):.3g}{bs}{extra}")
    # H3 frontier
    fam = [n for n in ("LoRA-match", "LoRA-match-sched", "LoRA-match-x0.5", "LoRA-match-x0.25") if n in R]
    if "U" in R and len(fam) >= 2:
        pts = sorted([(np.mean([R[n][s]["plasticity"] for s in R[n]]), np.mean([R[n][s]["forgetting"] for s in R[n]])) for n in fam])
        up = np.mean([R["U"][s]["plasticity"] for s in R["U"]]); uf = np.mean([R["U"][s]["forgetting"] for s in R["U"]])
        ff = float(np.interp(up, [p for p, _ in pts], [f for _, f in pts]))
        print(f"\nH3 frontier: at U's plasticity {up:.4f} the LoRA frontier forgets {ff:+.4f}; U forgets {uf:+.4f} -> {'below (pass)' if uf < ff else 'not below'}")
    if all(n in R for n in ("U", "U-K4", "LoRA-match")):
        f = lambda n: np.mean([R[n][s]["final"] for s in R[n]]); gain = f("LoRA-match") - f("U")
        if gain > 0: print(f"H4 bounded size: U-K4 keeps {(f('LoRA-match') - f('U-K4')) / gain:.0%} of U's final-loss improvement over LoRA-match (needs >= 70%)")
        else: print(f"H4 bounded size: undefined, because U's final loss is not below LoRA-match's ({gain:+.4f}); U-K4 - U = {f('U-K4') - f('U'):+.4f}")
    if "U" in R and "LoRA-match" in R and R["U"][next(iter(R["U"]))]["general1"]:
        d = lambda n, k, sgn: np.mean([sgn * (R[n][s]["general1"][k] - R[n][s]["general0"][k]) for s in R[n]])
        print(f"H5 general ability: LAMBADA drop U {d('U', 'lambada_acc', -1):.3f} vs LoRA-match {d('LoRA-match', 'lambada_acc', -1):.3f}; "
              f"WikiText rise U {d('U', 'wikitext_loss', 1):.3f} vs {d('LoRA-match', 'wikitext_loss', 1):.3f}")
    # secondary: creation, routing, recurrence
    S = CFG["steps_per_segment"]
    for n in [n for n in names if CONDITIONS[n]["kind"] == "pop" and not CONDITIONS[n].get("oracle")]:
        for s in sorted(R[n], key=int):
            r = R[n][s]; nseg = len(r["segments"]); cr = [e[0] for e in r["events"] if e[1] == "create"]
            hit = sum(any(k * S < c <= k * S + 50 for c in cr) for k in range(1, nseg - 1))
            spur = sum(not any(k * S < c <= k * S + 50 for k in range(0, nseg)) for c in cr)
            rec = any((nseg - 1) * S < c <= nseg * S for c in cr); rc = r.get("recurrence", {})
            print(f"  {n:14s} s{s}: creation at {hit}/{nseg - 2} switches, {spur} spurious, new part on recurrence: {rec}, "
                  f"final ARI {r['stages'][-1]['ari']:.2f}, recurrence loss {rc.get('first', np.nan):.3f} -> {rc.get('after', np.nan):.3f}")
    # plots
    fig, ax = plt.subplots(figsize=(7, 4.8))
    for n in names:
        p = _tci([R[n][s]["plasticity"] for s in R[n]]); f = _tci([R[n][s]["forgetting"] for s in R[n]])
        ax.errorbar(p[0], f[0], xerr=0 if np.isnan(p[2]) else p[2], yerr=0 if np.isnan(f[2]) else f[2], fmt="o", capsize=3, label=n)
    ax.set_xlabel("plasticity (nats/token, lower = better)"); ax.set_ylabel("forgetting (nats/token, lower = better)")
    ax.grid(alpha=.3); ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
    pn = [n for n in ("U", "BAL", "U-K4") if n in R]
    if pn:
        fig, axes = plt.subplots(1, len(pn), figsize=(5.2 * len(pn), 4), squeeze=False)
        for ax, n in zip(axes[0], pn):
            r = R[n]["0"]; st = r["stages"][len(r["order"]) - 1]; Pm = np.array([st["post"][d] for d in r["order"]])
            im = ax.imshow(Pm, vmin=0, vmax=1, aspect="auto"); ax.set_yticks(range(len(r["order"]))); ax.set_yticklabels(r["order"], fontsize=7)
            ax.set_xticks(range(len(st["ex"]))); ax.set_xticklabels([f"{c}\n{st['mass'][c]:.2f}" for c in st["ex"]], fontsize=6)
            ax.set_title(f"{n} seed 0: routing before the recurrence (ARI {st['ari']:.2f})", fontsize=8)
        fig.colorbar(im, ax=axes[0].tolist(), shrink=.8); plt.show()
    tl = [n for n in ("U", "BAL", "U-K4", "U-gradual") if n in R]
    if tl:
        fig, ax = plt.subplots(figsize=(9, 0.6 + 0.45 * len(tl)))
        for i, n in enumerate(tl):
            r = R[n][sorted(R[n], key=int)[0]]
            for e in r["events"]:
                if e[1] in ("create", "promote", "merge", "pool full"):
                    ax.scatter(e[0], i, marker={"create": "o", "promote": "|", "merge": "x", "pool full": "s"}[e[1]],
                               color={"create": "C0", "promote": "C2", "merge": "C3", "pool full": "k"}[e[1]], s=40)
        for k in range(1, len(r["segments"])): ax.axvline(k * S, color="grey", lw=.6, ls=":")
        ax.set_yticks(range(len(tl))); ax.set_yticklabels(tl); ax.set_xlabel("step (dotted = domain switch, unknown to the model)")
        ax.set_title("creation (o), promotion (|), merge (x), pool full (square): first seed", fontsize=9); plt.tight_layout(); plt.show()
    if "Single-domain" in R and "U" in R:
        sdm = R["Single-domain"]["0"]; fig, ax = plt.subplots(figsize=(7, 3.5))
        for n in [n for n in ("U", "LoRA-match", "BAL", "Full-FT") if n in R]:
            r = R[n]["0"]; M = np.array(r["M"]); T = len(r["order"])
            ax.plot(range(1, T + 1), [M[t + 1, t] - sdm[d] for t, d in enumerate(r["order"])], marker="o", label=n)
        ax.axhline(0, color="k", lw=.8); ax.set_xlabel("segment"); ax.set_ylabel("loss right after learning - fresh model (nats)")
        ax.set_title("loss of plasticity along the stream (seed 0)", fontsize=9); ax.grid(alpha=.3); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

report()

## Reading the results
* **H1 (the method's core claim):** U forgets at most 70% of what BAL forgets. U and BAL share architecture, E-step, creation and promotion; only the normalisation differs.
* **H2a/H2b:** U's final loss beats parameter-matched LoRA and the learned router. If H1 holds but H2 fails, the method prevents forgetting but does not pay at this scale; report that.
* **H3:** U lies below the LoRA forgetting–plasticity frontier (learning-rate scaling and U's own schedule) at U's plasticity, i.e. it is more than "learn slower".
* **H4:** U-K4 keeps at least 70% of U's improvement with a third of the components.
* **H5:** U harms general ability no more than LoRA-match.
* **Recurrence:** the 13th segment should create no new component, and the recurring domain's loss should come back quickly.
* Every failed hypothesis is reported as such. Deviations from the pre-registration go in a separate, labelled section of the paper.